In [1]:
# from datautils import downloader
# in case of using colab or any other online platform run this cell

# !git clone https://github.com/Pooria90/EEG-Motor-Imagery-Analysis.git
import os
os.chdir('/Users/rishabhkumar/Kernel-Experiments/betaprime-experiments')

# path  = 'data'
# dpath = downloader(path, subjects=list(range(1,10)))
path = '/Users/rishabhkumar/mne_data/MNE-bnci-data/database/data-sets/001-2014'

# !echo "Current working directory: $(pwd)"
from datautils import mat_extractor

tr_name = 'A01T.mat'
te_name = 'A01E.mat'

x_train, y_train = mat_extractor(path=path + '/' + tr_name, bpf_dict={'apply': True, 'fs': 250, 'lc':8, 'hc':35, 'order':5})
x_test , y_test  = mat_extractor(path=path + '/' + te_name)

print (f'*** Shapes ***\nx_train:\t{x_train.shape}\ny_train:\t{y_train.shape}')
print (f'x_test:\t\t{x_test.shape}\ny_test:\t\t{y_test.shape}')

/Users/rishabhkumar/miniconda3/envs/bci_env/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


*** Shapes ***
x_train:	(288, 22, 1000)
y_train:	(288,)
x_test:		(288, 22, 1000)
y_test:		(288,)


In [3]:
import numpy as np
from scipy.linalg import fractional_matrix_power, logm, expm

def arithmetic_mean(covmats):
    """Compute the arithmetic mean of covariance matrices."""
    return np.mean(covmats, axis=0)

def geometric_mean(covmats, max_iter=50, tol=1e-9):
    """Compute the geometric (Riemannian) mean of covariance matrices."""
    n_matrices = covmats.shape[0]
    mean = arithmetic_mean(covmats)
    for _ in range(max_iter):
        logs = np.array([logm(np.dot(fractional_matrix_power(mean, -0.5), 
                                    np.dot(cov, fractional_matrix_power(mean, -0.5))))
                         for cov in covmats])
        delta = np.mean(logs, axis=0)
        mean = np.dot(fractional_matrix_power(mean, 0.5), 
                      np.dot(expm(delta), fractional_matrix_power(mean, 0.5)))
        if np.linalg.norm(delta) < tol:
            break
    return mean

In [4]:
# Calculate the covariance matrices for the signal 
from datautils import multi_signal_cov_calculator

cov_train = multi_signal_cov_calculator(x_train, type='oas')

In [5]:
# test arithmetic mean of covariance matrices
cov_arithmetic_mean = arithmetic_mean(cov_train)
print(f'Arithmetic Mean Covariance Matrix:\n{cov_arithmetic_mean}')

Arithmetic Mean Covariance Matrix:
[[0.96829586 0.8199511  0.98245485 1.06511334 0.99526141 0.86136914
  0.42817033 0.64589804 0.65466076 0.65619169 0.72304908 0.67759437
  0.58539256 0.18049107 0.1276383  0.25106097 0.49387117 0.53839999
  0.42808079 0.85940246 0.90784643 0.88190603]
 [0.8199511  0.81258491 0.90659223 0.93923809 0.84278744 0.73922925
  0.48349874 0.69400462 0.66073722 0.6241556  0.64116239 0.59966921
  0.5002343  0.28621168 0.21710179 0.30977684 0.48538319 0.52115089
  0.48855098 0.83907208 0.87389599 0.86409325]
 [0.98245485 0.90659223 1.06811466 1.11792869 1.02398946 0.89087391
  0.5278947  0.7732576  0.76864621 0.72872595 0.77787625 0.71962412
  0.61062301 0.28518445 0.22986374 0.35053284 0.58363927 0.63916685
  0.57632116 1.02193327 1.07875214 1.06723113]
 [1.06511334 0.93923809 1.11792869 1.22870624 1.13040779 0.99162211
  0.50638839 0.77943484 0.77725258 0.78691977 0.84689064 0.80496997
  0.67413595 0.25640284 0.18647402 0.34717431 0.61554341 0.67026377
  0.5637

In [2]:
# use tangent space at 
# 1. compute the geometric mean of covariance matrices for each subject
# 2. use the tangent space at the geometric mean to compute the covariance matrices

cov_geometric_mean = geometric_mean(cov_train)
print(f'Geometric Mean Covariance Matrix:\n{cov_geometric_mean}')
# tangent space at the geometric mean
cov_tangent_space = np.array([np.dot(fractional_matrix_power(cov_geometric_mean, -0.5), 
                                      np.dot(cov, fractional_matrix_power(cov_geometric_mean, -0.5)))
                              for cov in cov_train])

print(f'Tangent Space Covariance Matrices:\n{cov_tangent_space}')
# Check the shape of the tangent space covariance matrices
print(f'Shape of Tangent Space Covariance Matrices: {cov_tangent_space.shape}')

NameError: name 'geometric_mean' is not defined

## Aligning Trials to 
1. Euclidean mean (Arithmetic Mean)
2. Geometric Mean

In [6]:
import numpy as np
from pyriemann.estimation import Covariances
from pyriemann.utils.mean import mean_covariance
from pyriemann.utils.tangentspace import tangent_space

In [8]:
# X shape: (n_trials, n_channels, n_times)
covs = Covariances(estimator='scm').fit_transform(x_train)  
# covs shape: (n_trials, n_channels, n_channels)

In [11]:
ref_mean = mean_covariance(covs, metric='riemann')
print(f'Reference Mean Covariance Matrix:\n{ref_mean.shape}')

Reference Mean Covariance Matrix:
(22, 22)


In [12]:
# Returns array of shape (n_trials, n_channels*(n_channels+1)/2)
tangent_feats = tangent_space(covs, ref_mean, metric='riemann')